# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
dataset_title = getattr(dataset.metadata, 'name', '')
dataset_description = getattr(dataset.metadata, 'description', '')
print(f"{dataset_title}: {dataset_description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all the `recordSet`, along with their `@id`, and their contained `fields` with their own `@id`s.

In [ ]:
# Examine available record sets and their fields
record_sets = getattr(dataset.metadata, 'recordSet', [])
print('Record Sets Found:')
for i, rs in enumerate(record_sets):
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"{i}. RecordSet @id: {rs_id}  |  Name: {rs_name}")
    
    # List fields for each record set
    fields = getattr(rs, 'field', [])
    if hasattr(fields, '__iter__') and not isinstance(fields, str):
        for fld in fields:
            fld_id = getattr(fld, '@id', None)
            fld_name = getattr(fld, 'name', None)
            print(f"    - Field @id: {fld_id}  |  Name: {fld_name}")
    else:
        fld = fields
        if fld:
            fld_id = getattr(fld, '@id', None)
            fld_name = getattr(fld, 'name', None)
            print(f"    - Field @id: {fld_id}  |  Name: {fld_name}")
print("\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Let's extract all record sets into pandas DataFrames using their `@id` for programmatic access.

In [ ]:
# Prepare list of RECORD SET IDs for extraction
record_set_ids = [getattr(rs, '@id', None) for rs in getattr(dataset.metadata, 'recordSet', [])]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet: {record_set_id} (rows: {len(df)}, cols: {len(df.columns)})")
    else:
        print(f"No records for RecordSet: {record_set_id}")

# Show the first record set's columns and head
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"\nColumns in {primary_rs_id}:")
    if primary_rs_id in dataframes:
        print(dataframes[primary_rs_id].columns.tolist())
        display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.
**Note:** Example uses the first record set. Adjust the variable names and field `@id` as needed.

In [ ]:
# If no record sets with data, skip this section
if dataframes:
    from IPython.display import display
    # Select the main record set to work with
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Try to auto-detect the first numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to cast columns to numeric
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notna().sum() > 0:
                df[col] = coerced
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break

    print(f"Chosen numeric field for analysis: {numeric_field}")

    if numeric_field:
        threshold = df[numeric_field].quantile(0.9)  # 90th percentile cutoff for demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (Top 10%):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by the next column if available
        group_field = None
        for col in df.columns:
            if col != numeric_field:
                group_field = col
                break
        print(f"Grouping field: {group_field}")
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped average of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    # Histogram of the chosen numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group field if detected
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} distribution by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- Explored the record sets and main fields of the dataset using their `@id`s.
- Loaded the primary record set into a DataFrame for flexible manipulation.
- Detected and analyzed a numeric field, applying threshold filtering and normalization.
- Visualized data distributions and group-wise summaries.

Further analyses can be performed by selecting specific record sets or fields by their `@id`, leveraging `mlcroissant` and pandas functionality.